In [ ]:
# v4: 2-step training — Step1 pretrain on full 83K rows, Step2 finetune on clean 4.8K rows
# Motivation: clean-data-only severely overfits; 2-step preserves minority class knowledge
# Architecture: EfficientNet-B0 + KL loss (same as v3)
# v6: 2D CNN on spectrograms — EfficientNet-B0 + KL loss + early stopping
!pip install timm -q

In [ ]:
import sys, os, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

# ── src imports from uploaded code dataset ──────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    CODE_DIR = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code'
else:
    CODE_DIR = os.path.abspath('../kaggle_upload')

sys.path.insert(0, CODE_DIR)
sys.path.insert(0, os.path.join(CODE_DIR, 'src'))

from src.config_2d  import Config2D
from src.dataset_2d import SpectrogramDataset
from src.model_2d   import EfficientNetEEG

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | torch {torch.__version__}")

In [ ]:
cfg = Config2D()

if IS_KAGGLE:
    DATA_ROOT = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    # train_test_split.csv (not train.csv) — contains inner_fold and split columns
    cfg.metadata_csv    = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_test_split.csv'
    cfg.spectrogram_dir = os.path.join(DATA_ROOT, 'train_spectrograms')
else:
    DATA_ROOT = os.path.abspath('../')
    cfg.metadata_csv    = os.path.abspath('../data_meta_splits/train_test_split.csv')
    cfg.spectrogram_dir = os.path.join(DATA_ROOT, 'train_spectrograms')

# ── reproducibility ─────────────────────────────────────────────────────────
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE  = torch.device(cfg.device)
USE_AMP = DEVICE.type == 'cuda'

print(cfg)
print(f"Device: {DEVICE} | AMP: {USE_AMP}")

In [ ]:
import sys
sys.path.append('/kaggle/input/hms-eeg-code/')
from src.data import load_clean_data, load_full_data

CLEAN_PATH = '/kaggle/input/hms-eeg-code/train_clean.csv'
FULL_PATH  = '/kaggle/input/hms-eeg-code/train_test_split.csv'

# 2-step data metadata (for size reporting)
full_train_df  = load_full_data(FULL_PATH,  split='trainval')  # 83K rows, Step1
clean_train_df = load_clean_data(CLEAN_PATH, split='trainval') # 4.8K rows, Step2
val_df         = load_clean_data(CLEAN_PATH, split='test')     # 1.1K rows, shared

print(f'Step1 train: {len(full_train_df):,} rows')
print(f'Step2 train: {len(clean_train_df):,} rows')
print(f'Val/test   : {len(val_df):,} rows')


In [ ]:
model = EfficientNetEEG(
    backbone    = cfg.backbone,
    num_classes = cfg.num_classes,
    pretrained  = cfg.pretrained,
).to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone    : {cfg.backbone}")
print(f"Parameters  — total: {total_params:,} | trainable: {trainable_params:,}")

In [ ]:
# v4: 2-step training — EfficientNet-B0, KL loss, early stopping
GRAD_CLIP = 1.0

# 2-step hyperparameters
STEP1_EPOCHS = 20
STEP2_EPOCHS = 30
STEP1_LR     = 1e-3
STEP2_LR     = 1e-4   # smaller LR for finetune

criterion     = nn.KLDivLoss(reduction='batchmean')
val_criterion = nn.CrossEntropyLoss()
scaler        = torch.amp.GradScaler('cuda', enabled=USE_AMP)

history = {'step1': [], 'step2': []}

# val_loader: inner_fold split from clean trainval (~946 rows)
# SpectrogramDataset always filters to trainval; test rows accessed via inner_fold
_val_ds    = SpectrogramDataset(CLEAN_PATH, cfg.spectrogram_dir, cfg, split='val')
val_loader = DataLoader(_val_ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True)
print(f'Val loader: {len(_val_ds):,} samples')


def validate_2d(model, loader):
    model.eval()
    val_ce, val_kl = 0.0, 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            x    = batch['image'].to(DEVICE)
            y    = batch['label'].to(DEVICE)
            soft = batch['soft_label'].to(DEVICE)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = model(x)
            val_ce += val_criterion(logits, y).item()
            val_kl += criterion(F.log_softmax(logits, dim=1), soft).item()
            all_preds .extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(y.cpu().tolist())
    return (val_kl / len(loader),
            val_ce / len(loader),
            f1_score(all_labels, all_preds, average='macro', zero_division=0))


def train_phase_2d(phase_name, csv_path, epochs, lr, save_path, hist_key):
    # One training phase using SpectrogramDataset(split='all'). Mutates model in-place.
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    best_f1, best_epoch, wait = 0.0, 0, 0
    best_state = None

    phase_ds     = SpectrogramDataset(csv_path, cfg.spectrogram_dir, cfg, split='all')
    phase_loader = DataLoader(phase_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True)
    print(f'  {phase_name} train: {len(phase_ds):,} samples')

    for epoch in range(1, epochs + 1):
        # train
        model.train()
        t0 = time.time()
        train_loss = 0.0
        for batch in phase_loader:
            x    = batch['image'].to(DEVICE)
            soft = batch['soft_label'].to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits    = model(x)
                loss      = criterion(F.log_softmax(logits, dim=1), soft)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        scheduler.step()

        # validate
        val_kl, val_ce, macro_f1 = validate_2d(model, val_loader)
        avg_train = train_loss / len(phase_loader)
        elapsed   = time.time() - t0

        history[hist_key].append({
            'epoch': epoch, 'train_kl': avg_train,
            'val_kl': val_kl, 'val_ce': val_ce, 'macro_f1': macro_f1,
        })
        print(f'  Epoch {epoch:03d} | train_kl {avg_train:.4f} | '
              f'val_kl {val_kl:.4f} | val_ce {val_ce:.4f} | '
              f'macro_f1 {macro_f1:.4f} | {elapsed:.0f}s')

        if macro_f1 > best_f1:
            best_f1, best_epoch, wait = macro_f1, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(model.state_dict(), save_path)
            print(f'    ✓ saved {save_path} (f1={best_f1:.4f})')
        else:
            wait += 1
            if wait >= cfg.patience:
                print(f'  Early stopping at epoch {epoch}')
                break

    print(f'  {phase_name} done — best macro_f1={best_f1:.4f} at epoch {best_epoch}')
    if best_state is not None:
        model.load_state_dict(best_state)


# Step 1: pretrain on full data
print('=' * 50)
print(f'Step 1: pretrain on full data ({len(full_train_df):,} rows)')
print('=' * 50)
train_phase_2d('Step 1', FULL_PATH, STEP1_EPOCHS, STEP1_LR,
               'best_v4_2d_step1.pt', 'step1')

# Step 2: finetune on clean data
print('\n' + '=' * 50)
print(f'Step 2: finetune on clean data ({len(clean_train_df):,} rows)')
print('=' * 50)
train_phase_2d('Step 2', CLEAN_PATH, STEP2_EPOCHS, STEP2_LR,
               'best_v4_2d_step2.pt', 'step2')

print(f'\n2-step training complete.')
print(f'  Step1 best f1: {max(h["macro_f1"] for h in history["step1"]):.4f}')
print(f'  Step2 best f1: {max(h["macro_f1"] for h in history["step2"]):.4f}')


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for row, (phase_key, phase_label) in enumerate([('step1', 'Step1 (full data)'), ('step2', 'Step2 (clean finetune)')]):
    hist = history[phase_key]
    if not hist:
        continue
    epochs_   = [h['epoch']    for h in hist]
    train_kl_ = [h['train_kl'] for h in hist]
    val_kl_   = [h['val_kl']   for h in hist]
    val_ce_   = [h['val_ce']   for h in hist]
    macro_f1_ = [h['macro_f1'] for h in hist]

    ax1, ax2, ax3 = axes[row]
    ax1.plot(epochs_, train_kl_, label='train KL')
    ax1.plot(epochs_, val_kl_,   label='val KL')
    ax1.set_title(f'KL Divergence — {phase_label}'); ax1.legend()

    ax2.plot(epochs_, val_ce_, color='orange', label='val CE')
    ax2.set_title(f'Val CE — {phase_label}'); ax2.legend()

    ax3.plot(epochs_, macro_f1_, color='green')
    ax3.set_title(f'Val Macro F1 — {phase_label}')

plt.tight_layout()
plt.savefig('training_curves_v4_2d.png', dpi=150)
plt.show()

for phase_key, phase_label in [('step1', 'Step1'), ('step2', 'Step2')]:
    hist = history[phase_key]
    if hist:
        f1s = [h['macro_f1'] for h in hist]
        print(f'{phase_label} best macro_f1: {max(f1s):.4f} (epoch {f1s.index(max(f1s))+1})')
